# FAERS Signal Detector Build + EDA Notebook

This notebook captures an end-to-end, reproducible build workflow from the project specification.

## Outline
1. Initialize workspace and project scaffold
2. Write `requirements.txt`, `.env.example`, `.gitignore`
3. Create reference CSV files
4. Create core source files (`config.py`, `db.py`)
5. Create SQL files
6. Implement download module
7. Implement ingestion module
8. Implement cleaning module
9. Implement signal computation module
10. Implement trend utilities
11. Create dashboard files
12. Create unit tests
13. Generate minimal EDA notebook programmatically
14. Write README and run validation commands

In [ ]:
from pathlib import Path

root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
project = root

expected_dirs = [
    'data/raw', 'data/processed', 'data/reference',
    'src', 'sql', 'dashboard/pages', 'dashboard/components',
    'notebooks', 'tests'
]
for d in expected_dirs:
    (project / d).mkdir(parents=True, exist_ok=True)

for g in ['data/raw/.gitkeep', 'data/processed/.gitkeep']:
    (project / g).touch(exist_ok=True)

missing = [d for d in expected_dirs if not (project / d).exists()]
assert not missing, f'Missing directories: {missing}'
print('Scaffold check passed')

In [ ]:
requirements_text = '''# Core data
pandas==2.2.2
numpy==1.26.4
sqlalchemy==2.0.30
psycopg2-binary==2.9.9

# Data cleaning
rapidfuzz==3.9.3
regex==2024.5.15

# HTTP / API
requests==2.32.3
tqdm==4.66.4

# Dashboard
streamlit==1.35.0
plotly==5.22.0

# Stats
scipy==1.13.1

# Dev / testing
pytest==8.2.2
python-dotenv==1.0.1
jupyter==1.0.0
ipykernel==6.29.4
'''

env_text = '''DB_HOST=localhost
DB_PORT=5432
DB_NAME=faers
DB_USER=postgres
DB_PASSWORD=yourpassword
FAERS_START_YEAR=2022
FAERS_END_YEAR=2024
FAERS_QUARTERS=1,2,3,4
'''

gitignore_text = '''data/raw/
data/processed/
.env
__pycache__/
*.pyc
.DS_Store
*.egg-info/
.pytest_cache/
'''

(project / 'requirements.txt').write_text(requirements_text, encoding='utf-8')
(project / '.env.example').write_text(env_text, encoding='utf-8')
(project / '.gitignore').write_text(gitignore_text, encoding='utf-8')

assert (project / 'requirements.txt').read_text(encoding='utf-8') == requirements_text
assert (project / '.env.example').read_text(encoding='utf-8') == env_text
assert (project / '.gitignore').read_text(encoding='utf-8') == gitignore_text
print('Config files check passed')

In [ ]:
import pandas as pd

brand_path = project / 'data/reference/brand_to_generic.csv'
soc_path = project / 'data/reference/meddra_soc.csv'

brand_df = pd.read_csv(brand_path)
soc_df = pd.read_csv(soc_path)

assert {'brand', 'generic'} == set(brand_df.columns)
assert {'reaction', 'soc_name'} == set(soc_df.columns)
assert (brand_df.loc[brand_df['brand'] == 'TYLENOL', 'generic'].iloc[0] == 'ACETAMINOPHEN')
assert (soc_df.loc[soc_df['reaction'] == 'DEATH', 'soc_name'].iloc[0] == 'General Disorders')
print(f'Brand mappings: {len(brand_df)} | SOC mappings: {len(soc_df)}')

## Core Modules, SQL, and Dashboard Validation

This project writes all code files under `src/`, `sql/`, and `dashboard/` from the specification. The checks below validate structure and critical content markers for:
- `src/config.py`, `src/db.py`, `src/download.py`, `src/ingest.py`, `src/clean.py`, `src/signals.py`, `src/trends.py`
- `sql/create_tables.sql`, `sql/signal_query.sql`
- `dashboard/app.py`, `dashboard/pages/*.py`, `dashboard/components/*.py`
- `tests/test_signals.py`, `tests/test_clean.py`, `README.md`

In [ ]:
required_files = [
    'src/config.py', 'src/db.py', 'src/download.py', 'src/ingest.py', 'src/clean.py', 'src/signals.py', 'src/trends.py',
    'sql/create_tables.sql', 'sql/signal_query.sql',
    'dashboard/app.py',
    'dashboard/pages/01_Drug_Explorer.py', 'dashboard/pages/02_Signal_Trends.py', 'dashboard/pages/03_Severity_Filter.py',
    'dashboard/components/signal_table.py', 'dashboard/components/trend_chart.py',
    'tests/test_signals.py', 'tests/test_clean.py', 'README.md'
]

missing_files = [f for f in required_files if not (project / f).exists()]
assert not missing_files, f'Missing files: {missing_files}'

ddl = (project / 'sql/create_tables.sql').read_text(encoding='utf-8')
assert 'CREATE TABLE IF NOT EXISTS raw_demo' in ddl
assert 'CREATE TABLE IF NOT EXISTS signal_results' in ddl
assert 'CREATE INDEX IF NOT EXISTS idx_signal_is_signal' in ddl

signal_sql = (project / 'sql/signal_query.sql').read_text(encoding='utf-8')
assert 'WITH' in signal_sql and 'drug_event AS' in signal_sql and 'ORDER BY prr DESC' in signal_sql

for page in ['dashboard/pages/01_Drug_Explorer.py', 'dashboard/pages/02_Signal_Trends.py', 'dashboard/pages/03_Severity_Filter.py']:
    text = (project / page).read_text(encoding='utf-8').lstrip()
    assert 'st.set_page_config' in text

print('File and static-content checks passed')

In [ ]:
from src.clean import normalize_age, normalize_drug_name
from src.signals import compute_contingency

brand_map = {'TYLENOL': 'ACETAMINOPHEN', 'LIPITOR': 'ATORVASTATIN'}
generics = list(brand_map.values())

assert normalize_age('18', 'MON') == 1.5
assert normalize_drug_name('Tylenol 500mg', brand_map, generics) == 'ACETAMINOPHEN'

result = compute_contingency(a=50, b=50, c=10, d=890)
assert result is not None and result['prr'] > 4 and result['is_signal']
print('Core math and cleaning smoke checks passed')

In [ ]:
import nbformat as nbf

eda_nb = nbf.v4.new_notebook()
eda_nb.cells = [
    nbf.v4.new_markdown_cell('# FAERS EDA Notes'),
    nbf.v4.new_markdown_cell('## Raw Tables\nDEMO, DRUG, REAC, OUTC, INDI from FAERS quarterly exports.'),
    nbf.v4.new_markdown_cell('## Join Strategy\nUse `primaryid` joins across cleaned tables for signal analysis.'),
    nbf.v4.new_markdown_cell('## Planned Analyses\nCounts by drug/event, PRR distribution, SOC breakdowns.'),
    nbf.v4.new_code_cell('print("Placeholder: load summary counts from PostgreSQL")'),
]
nbf.write(eda_nb, project / 'notebooks/01_EDA.ipynb')
print('Minimal EDA notebook generated/updated')

In [ ]:
import subprocess

RUN_HEAVY = False

commands = [
    'pip install -r requirements.txt',
    'createdb faers',
    'python pipeline.py --all',
    'pytest tests/ -v',
    'streamlit run dashboard/app.py'
]
print('Planned validation commands:')
for cmd in commands:
    print('-', cmd)

if RUN_HEAVY:
    for cmd in commands[:-1]:
        print('\nRunning:', cmd)
        proc = subprocess.run(cmd, shell=True, cwd=project, capture_output=True, text=True)
        print('returncode =', proc.returncode)
        print(proc.stdout[-2000:])
        print(proc.stderr[-2000:])
else:
    print('RUN_HEAVY=False, heavy commands skipped intentionally.')